# Local GUNW + DEM processing smoke check

This notebook exercises SnowIn on a real local NISAR GUNW and an explicitly declared prepared DEM. It opens the product, finds a 128 × 128 window with strong DEM support inside the GUNW grid, computes terrain-surface incidence, and retrieves pairwise dSWE. It does not download or write product data.

Set `SNOWIN_GUNW`, `SNOWIN_NISAR_DEM`, and `SNOWIN_DEM_SOURCE` in the environment or the ignored repository-root `.env` file. The source declaration must match the DEM. Ellipsoidal matching can be required for ellipsoidal DEM sources; orthometric inputs run provisionally when no vertical correction is supplied.

## Local inputs and interpretation

The notebook reads `SNOWIN_GUNW`, `SNOWIN_NISAR_DEM`, and `SNOWIN_DEM_SOURCE`
from the process environment or an ignored repository-root `.env` file. Without
those settings it prompts for the paths and the DEM source. The smoke window is
chosen from the projected DEM/GUNW overlap. COP30 and SRTM30 inputs are
orthometric; without a supplied vertical correction their geometry is marked
`mismatch_not_corrected`, so the dSWE result checks software execution and is
not a datum-corrected scientific estimate.


In [ ]:
import os
from pathlib import Path

import numpy as np
import rasterio
from rasterio.crs import CRS
from rasterio.warp import transform_bounds

from snowin import compute_dswe
from snowin.io import add_gunw_incidence, open_gunw


ROOT_CANDIDATES = (Path.cwd(), *Path.cwd().parents)
REPO_ROOT = next(
    (candidate for candidate in ROOT_CANDIDATES if (candidate / "pyproject.toml").is_file()),
    Path.cwd(),
)


def load_local_env():
    env_path = REPO_ROOT / ".env"
    if not env_path.is_file():
        return
    allowed_keys = {"SNOWIN_GUNW", "SNOWIN_NISAR_DEM", "SNOWIN_DEM_SOURCE"}
    for raw_line in env_path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        key, separator, value = line.partition("=")
        key = key.strip()
        if separator and key in allowed_keys:
            os.environ.setdefault(key, value.strip().strip("\"'"))


load_local_env()


def local_path_from_env(variable, label):
    value = os.environ.get(variable, "").strip()
    if not value:
        value = input(f"Path to local {label}: ").strip()
    if not value:
        raise RuntimeError(f"A path for {label} is required.")
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = REPO_ROOT / path
    path = path.resolve()
    if not path.is_file():
        raise FileNotFoundError(
            f"{variable} does not name an existing file: {path}. "
            "Use an absolute path or a repository-relative path."
        )
    return path


gunw_path = local_path_from_env("SNOWIN_GUNW", "GUNW")
dem_path = local_path_from_env("SNOWIN_NISAR_DEM", "prepared DEM")
dem_source = os.environ.get("SNOWIN_DEM_SOURCE", "").strip().lower()
if not dem_source:
    dem_source = input(
        "DEM source (nisar_cop30, cop30, tandem30, srtm30): "
    ).strip().lower()
if dem_source not in {"nisar_cop30", "cop30", "tandem30", "srtm30"}:
    raise ValueError(
        "Set SNOWIN_DEM_SOURCE to nisar_cop30, cop30, tandem30, or srtm30."
    )

print(f"GUNW: {gunw_path.name}")
print(f"DEM:  {dem_path.name}")
print(f"DEM source declaration: {dem_source}")

In [ ]:
with rasterio.open(dem_path) as dem_file:
    dem_tags = dem_file.tags()
    tagged_gunw = dem_tags.get("source_gunw")
    declared_gunw_match = (
        Path(tagged_gunw).name == gunw_path.name if tagged_gunw else None
    )
    dem_height_reference = dem_tags.get("height_reference", "").strip().lower()
    dem_summary = {
        "crs": dem_file.crs.to_string() if dem_file.crs else None,
        "shape_y_x": (dem_file.height, dem_file.width),
        "source_product": dem_tags.get("source_product"),
        "height_reference_tag": dem_tags.get("height_reference"),
        "declared_source_gunw_matches": declared_gunw_match,
    }

expected_height_references = {
    "nisar_cop30": {"ellipsoid", "ellipsoidal", "wgs84 ellipsoid"},
    "cop30": {"orthometric", "egm2008"},
    "tandem30": {"ellipsoid", "ellipsoidal", "wgs84 ellipsoid", "wgs84-g1150"},
    "srtm30": {"orthometric", "egm96"},
}
if declared_gunw_match is False:
    raise ValueError("The DEM metadata names a different source GUNW.")
if dem_height_reference and dem_height_reference not in expected_height_references[dem_source]:
    raise ValueError(
        f"DEM height_reference tag {dem_height_reference!r} conflicts with "
        f"SNOWIN_DEM_SOURCE={dem_source!r}."
    )
if dem_summary["crs"] is None:
    raise ValueError("The DEM must declare a coordinate reference system.")

print(dem_summary)
if declared_gunw_match is None:
    print("The DEM has no source_gunw tag; its pairing is supplied by local configuration.")
if dem_source in {"cop30", "srtm30"}:
    print(
        "This DEM source is orthometric. No vertical correction is supplied; "
        "local geometry is provisional and should report mismatch_not_corrected."
    )

In [ ]:
pair = open_gunw(gunw_path, chunks="auto", progress=True)
epsg_code = pair.spatial_ref.attrs.get("epsg_code")
print(f"Opened {gunw_path.name}")
print(f"Phase grid (y, x): {pair.phase.shape}")
print(f"Grid EPSG: {epsg_code}")
print(f"Wavelength: {float(pair.attrs['wavelength_m']):.6f} m")

The reader keeps GUNW arrays lazy when Dask is installed. The small window below bounds the phase and DEM calculation; SnowIn still reads its compact radar-grid LOS lookup cube to calculate local incidence.

In [ ]:
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from rasterio.warp import reproject

window_size = 128
height, width = pair.phase.shape
if height < window_size or width < window_size:
    raise ValueError("The GUNW phase grid is smaller than the requested test window.")

epsg_code = int(pair.spatial_ref.attrs["epsg_code"])
target_crs = CRS.from_epsg(epsg_code)
with rasterio.open(dem_path) as dem_file:
    if dem_file.crs is None:
        raise ValueError("The DEM must declare a coordinate reference system.")
    target_bounds = transform_bounds(
        dem_file.crs, target_crs, *dem_file.bounds, densify_pts=21
    )
    source_valid = np.isfinite(dem_file.read(1)).astype("float32")
    source_transform = dem_file.transform
    source_crs = dem_file.crs

left, bottom, right, top = target_bounds
x_coords = np.asarray(pair.coords["x"].data, dtype=float)
y_coords = np.asarray(pair.coords["y"].data, dtype=float)
columns = np.flatnonzero((x_coords >= left) & (x_coords <= right))
rows = np.flatnonzero((y_coords >= bottom) & (y_coords <= top))
if columns.size < window_size or rows.size < window_size:
    raise ValueError(
        "The DEM/GUNW overlap is smaller than the requested smoke window: "
        f"{rows.size} rows by {columns.size} columns."
    )
if np.any(np.diff(columns) != 1) or np.any(np.diff(rows) != 1):
    raise ValueError("The DEM/GUNW overlap is not contiguous on the phase grid.")

x_overlap = x_coords[columns]
y_overlap = y_coords[rows]
dx = float(np.median(np.abs(np.diff(x_overlap))))
dy = float(np.median(np.abs(np.diff(y_overlap))))
target_transform = from_origin(
    float(np.min(x_overlap) - dx / 2),
    float(np.max(y_overlap) + dy / 2),
    dx,
    dy,
)
support_weights = np.zeros((rows.size, columns.size), dtype="float32")
reproject(
    source_valid,
    support_weights,
    src_transform=source_transform,
    src_crs=source_crs,
    dst_transform=target_transform,
    dst_crs=target_crs,
    resampling=Resampling.bilinear,
)
support = support_weights > 0
integral = np.pad(support.astype("int32"), ((1, 0), (1, 0))).cumsum(0).cumsum(1)
window_support = (
    integral[window_size:, window_size:]
    - integral[:-window_size, window_size:]
    - integral[window_size:, :-window_size]
    + integral[:-window_size, :-window_size]
)
best_row, best_column = np.unravel_index(np.argmax(window_support), window_support.shape)
best_valid_fraction = float(window_support[best_row, best_column] / window_size**2)
if best_valid_fraction < 0.80:
    raise ValueError(
        "No 128x128 smoke window has at least 80% valid DEM support; "
        f"best available fraction is {best_valid_fraction:.3f}."
    )

row_start = int(rows[best_row])
column_start = int(columns[best_column])
subset = pair.isel(
    y=slice(row_start, row_start + window_size),
    x=slice(column_start, column_start + window_size),
).copy()

print(f"DEM/GUNW overlap: {rows.size} rows by {columns.size} columns")
print(f"Selected window valid DEM fraction: {best_valid_fraction:.3f}")
print(f"Test window shape: {subset.phase.shape}")
print(f"Window starts at phase-grid row {row_start}, column {column_start}.")

In [ ]:
require_vertical_datum_match = dem_source in {"nisar_cop30", "tandem30"}
add_gunw_incidence(
    subset,
    gunw_path,
    dem_source=dem_source,
    dem=dem_path,
    require_vertical_datum_match=require_vertical_datum_match,
    progress=True,
)

incidence_values = np.asarray(subset.incidence_angle.compute().data, dtype=float)
finite_incidence = np.isfinite(incidence_values)
assert subset.incidence_angle.attrs["incidence_angle_reference"] == "local"
assert finite_incidence.mean() >= 0.80, "Less than 80% of the smoke window has valid local incidence."
assert np.all(
    (incidence_values[finite_incidence] >= 0)
    & (incidence_values[finite_incidence] < np.pi / 2)
)
if dem_source in {"cop30", "srtm30"}:
    assert subset.attrs["vertical_datum_status"] == "mismatch_not_corrected"

incidence_degrees = np.rad2deg(incidence_values[finite_incidence])
print(f"Vertical datum status: {subset.attrs['vertical_datum_status']}")
print(f"Local incidence finite fraction: {finite_incidence.mean():.3f}")
print(
    "Local incidence degrees (min / median / max): "
    f"{np.percentile(incidence_degrees, 0):.2f} / "
    f"{np.median(incidence_degrees):.2f} / "
    f"{np.percentile(incidence_degrees, 100):.2f}"
)

In [ ]:
safe_incidence = subset.incidence_angle.where(subset.geometry_valid)
dswe = compute_dswe(
    subset.phase,
    safe_incidence,
    wavelength_m=float(pair.attrs["wavelength_m"]),
).compute()
phase_values = np.asarray(subset.phase.compute().data, dtype=float)
dswe_values_m = np.asarray(dswe.data, dtype=float)
valid = np.isfinite(phase_values) & np.isfinite(incidence_values) & np.isfinite(dswe_values_m)
assert dswe.dims == ("y", "x")
assert dswe.attrs.get("units") == "m", dswe.attrs.get("units")
assert valid.any(), "No valid dSWE pixels were produced."

dswe_values_mm = dswe_values_m[valid] * 1000.0
print(f"dSWE output shape: {dswe.shape}; units: {dswe.attrs['units']}")
print(f"Finite phase fraction: {np.isfinite(phase_values).mean():.3f}")
print(f"Finite dSWE fraction: {valid.mean():.3f}")
print(
    "dSWE millimeters (min / median / max): "
    f"{np.percentile(dswe_values_mm, 0):.3f} / "
    f"{np.median(dswe_values_mm):.3f} / "
    f"{np.percentile(dswe_values_mm, 100):.3f}"
)

In [ ]:
pair.close()
print("Closed the GUNW dataset.")